# Data Science with Deno 🦕

Deno runs TypeScript natively and can import packages straight from **npm** (`npm:`) and **JSR** (`jsr:`), with no `package.json` and no install step. In this short tutorial we'll:

1. Load a dataset into a **Polars** DataFrame
2. Explore, filter, and aggregate it
3. Visualize it with **Observable Plot**
4. Fit a simple regression model

First, check which Deno version the kernel is running:

In [ ]:
Deno.version

## 1. Load data into a DataFrame

[nodejs-polars](https://pola-rs.github.io/nodejs-polars/) brings the fast Rust-based Polars DataFrame library to JavaScript. We'll make a small synthetic dataset of coffee shop sales, write it to a CSV with Deno's built-in file APIs, and read it back.

In [ ]:
import pl from "npm:nodejs-polars";

// Generate a reproducible synthetic dataset
let seed = 42;
const rand = () => ((seed = (seed * 16807) % 2147483647) - 1) / 2147483646;

const shops = ["Downtown", "Campus", "Airport"];
const drinks = ["Espresso", "Latte", "Cold Brew", "Tea"];
const prices = { Espresso: 3.0, Latte: 4.5, "Cold Brew": 4.0, Tea: 2.5 };

const rows = [];
for (let day = 1; day <= 30; day++) {
  const temp = +(15 + 10 * Math.sin(day / 5) + rand() * 4).toFixed(1); // daily temperature (°C)
  for (const shop of shops) {
    for (const drink of drinks) {
      const coldBoost = drink === "Cold Brew" ? temp * 2 : 0;
      const units = Math.round(20 + rand() * 30 + coldBoost + (shop === "Airport" ? 15 : 0));
      rows.push({ day, shop, drink, temp, units, revenue: +(units * prices[drink]).toFixed(2) });
    }
  }
}

// Round-trip through a CSV file using Deno's file APIs
const header = Object.keys(rows[0]).join(",");
const csv = [header, ...rows.map((r) => Object.values(r).join(","))].join("\n");
await Deno.writeTextFile("coffee_sales.csv", csv);

const df = pl.readCSV(await Deno.readTextFile("coffee_sales.csv"));
df.head(5)

## 2. Explore, filter, and aggregate

**Summary statistics:**

In [ ]:
console.log("shape:", df.shape);
df.describe()

**Filter:**

In [ ]:
// Filter: Cold Brew sales on days warmer than 20°C
df.filter(pl.col("drink").eq(pl.lit("Cold Brew")).and(pl.col("temp").gt(20)))
  .select("day", "shop", "temp", "units")
  .head(5)

**Aggregate:** Polars expressions (`pl.col(...)`) chain together into fast, lazy-friendly queries, just like in Python.

In [ ]:
// Group by shop: total revenue and average units per row
df.groupBy("shop")
  .agg(
    pl.col("revenue").sum().round(2).alias("total_revenue"),
    pl.col("units").mean().round(1).alias("avg_units"),
  )
  .sort("total_revenue", true)

**Derived columns:** `pl.when/then/otherwise` works like SQL's `CASE WHEN`. Does weather change what people order?

In [ ]:
// Add a derived column, then pivot-style summary: average units per drink, hot vs. cool days
df
  .withColumns(pl.when(pl.col("temp").gt(20)).then(pl.lit("hot")).otherwise(pl.lit("cool")).alias("weather"))
  .groupBy(["drink", "weather"])
  .agg(pl.col("units").mean().round(1).alias("avg_units"))
  .sort(["drink", "weather"])

## 3. Visualize with Observable Plot

[Observable Plot](https://observablehq.com/plot/) is a concise grammar-of-graphics library, roughly what ggplot2 is in R. In the notebook we give it a lightweight DOM from `linkedom` and display the resulting SVG with `Deno.jupyter.html`.

In [ ]:
import * as Plot from "npm:@observablehq/plot";
import { parseHTML } from "npm:linkedom";

// Observable Plot builds SVG with a DOM; linkedom provides one outside the browser
const { document } = parseHTML("<html></html>");

const drinkColors = { domain: drinks, range: ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"], legend: true };

const daily = df.groupBy(["day", "drink"]).agg(pl.col("units").sum()).sort("day").toRecords();

const chart = Plot.plot({
  document,
  title: "Daily units sold by drink",
  width: 720,
  height: 320,
  color: drinkColors,
  x: { label: "Day" },
  y: { label: "Units sold", grid: true },
  marks: [
    Plot.lineY(daily, { x: "day", y: "units", stroke: "drink", strokeWidth: 2 }),
    Plot.text(daily, Plot.selectLast({ x: "day", y: "units", z: "drink", text: "drink", dx: 6, textAnchor: "start" })),
  ],
  marginRight: 70,
});

Deno.jupyter.html`${chart.outerHTML}`